# Chart Esai — ForestWatch Papua (skema final 5 kelas)

Baca langsung output training 7-kelas dari Google Drive (`metrics_finetune.json`,
`train_distribution.json`), gabungkan ke skema **5 kelas final** (Tambang -> Lahan Terbuka;
Sawit + Pertanian Lain -> Pertanian), lalu hasilkan semua chart untuk esai. Jalankan di Colab
dari atas ke bawah. Chart tersimpan PNG di Drive, folder yang sama dgn output training
(`.../Bahan_Training_Fix_Combined_v4/output/chart_esai/`).

Chart yang dihasilkan:
1. Distribusi kelas (ketidakseimbangan data) — motivasi metodologi
2. IoU per kelas (model final 5 kelas)
3. Confusion matrix 5 kelas (ternormalisasi)
4. Panel metrik utama (OA, Kappa, FWIoU, mIoU)
5. (opsional) IoU 7-kelas: sebelum vs sesudah fine-tune


In [ ]:
# === Bagian 0 -- Mount Drive + lokasi file output training 7-kelas ===
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

# SESUAIKAN kalau folder output-mu beda lokasi.
OUTPUT_DIR = Path("/content/drive/MyDrive/Satria Data 3.0/Bahan_Training_Fix_Combined_v4/output")
METRICS_JSON = OUTPUT_DIR / "metrics_finetune.json"
DIST_JSON    = OUTPUT_DIR / "train_distribution.json"

CHART_DIR = OUTPUT_DIR / "chart_esai"
CHART_DIR.mkdir(parents=True, exist_ok=True)

assert METRICS_JSON.exists(), f"Tak ditemukan: {METRICS_JSON}"
assert DIST_JSON.exists(), f"Tak ditemukan: {DIST_JSON}"
print("METRICS_JSON :", METRICS_JSON)
print("DIST_JSON    :", DIST_JSON)
print("Chart disimpan ke:", CHART_DIR)


In [ ]:
# === Bagian 1 -- Load data asli + setup ===
import json
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.axisbelow": True, "figure.dpi": 120,
                     "savefig.dpi": 150, "savefig.bbox": "tight"})

metrics7 = json.load(open(METRICS_JSON))
dist7_raw = json.load(open(DIST_JSON))
dist7 = {int(k): int(v) for k, v in dist7_raw.items()}

NAMES7 = ["Perairan", "Hutan", "Lahan Terbuka", "Sawit", "Pertanian Lain", "Tambang", "Permukiman"]
NAMES5 = ["Perairan", "Hutan", "Lahan Terbuka", "Pertanian", "Permukiman"]
COLORS5 = ["#2A6FDB", "#0B3D0B", "#E03B24", "#F97316", "#757575"]
REMAP = {0: 0, 1: 1, 2: 2, 3: 3, 4: 3, 5: 2, 6: 4}   # old 7-kelas -> new 5-kelas

cm7 = np.array(metrics7["confusion_matrix"], dtype=np.int64)
iou7_after = [r["iou"] for r in metrics7["per_class"]]

print(f"OA={metrics7['overall_accuracy']:.4f} | mIoU(7-kelas)={metrics7['mean_iou']:.4f} | "
      f"FWIoU={metrics7['fwiou']:.4f} | Kappa={metrics7['kappa']:.4f}")
print("Distribusi train (7-kelas mentah):", dist7)


In [ ]:
# === Bagian 2 -- (opsional) summary_finetune.json utk IoU SEBELUM fine-tune (chart 5) ===
SUMMARY_JSON = OUTPUT_DIR / "summary_finetune.json"
iou7_before = None
if SUMMARY_JSON.exists():
    summary7 = json.load(open(SUMMARY_JSON))
    iou7_before = summary7.get("baseline_test_per_class_iou")
    print("baseline_test_per_class_iou (sebelum fine-tune):", iou7_before)
else:
    print("summary_finetune.json tak ditemukan -- Chart 5 (before/after) akan dilewati.")


## Chart 1 — Distribusi kelas (ketidakseimbangan data)
Dipakai di **Pendahuluan/Metodologi** untuk menjelaskan tantangan kelas minoritas.

In [ ]:
# === Chart 1: Distribusi piksel kelas (train, gabung ke 5-kelas) ===
dist5 = {c: 0 for c in range(5)}
for old_c, px in dist7.items():
    dist5[REMAP[old_c]] += px

tot = sum(dist5.values())
pct = [100 * dist5[c] / tot for c in range(5)]
order = np.argsort(pct)  # kecil -> besar
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh([NAMES5[i] for i in order], [pct[i] for i in order],
        color=[COLORS5[i] for i in order], edgecolor="black", linewidth=0.5)
for i, idx in enumerate(order):
    ax.text(pct[idx] + 0.4, i, f"{pct[idx]:.1f}%", va="center", fontsize=10)
ax.set_xlabel("Proporsi piksel di data latih (%)")
ax.set_title("Distribusi Kelas Tutupan Lahan (5 kelas) - ketidakseimbangan kuat")
ax.set_xlim(0, max(pct) * 1.15); ax.grid(axis="y", visible=False)
fig.savefig(CHART_DIR / "01_distribusi_kelas.png"); plt.show()
print("Tersimpan:", CHART_DIR / "01_distribusi_kelas.png")


## Chart 2 — IoU per kelas (model final 5 kelas)
Chart performa utama. Dipakai di **Pembahasan** (hasil per kelas).

In [ ]:
# === Gabung confusion matrix 7-kelas -> 5-kelas + hitung metrik (dipakai chart 2-4) ===
cm5 = np.zeros((5, 5), dtype=np.int64)
for i in range(7):
    for j in range(7):
        cm5[REMAP[i], REMAP[j]] += cm7[i, j]

def iou_per_class(cm):
    d = np.diag(cm).astype(float); r = cm.sum(1); c = cm.sum(0)
    return d / (r + c - d)

iou5 = iou_per_class(cm5)
oa5 = float(np.diag(cm5).sum() / cm5.sum())
freq5 = cm5.sum(1) / cm5.sum()
fwiou5 = float((freq5 * iou5).sum())
miou5 = float(iou5.mean())
_n = cm5.sum(); _pe = (cm5.sum(0) * cm5.sum(1)).sum() / (_n * _n)
kappa5 = float((oa5 - _pe) / (1 - _pe))

print("PANEL METRIK 5-KELAS (TEST, dari penggabungan confusion matrix 7-kelas):")
print(f"  OA={oa5:.4f} | Kappa={kappa5:.4f} | mIoU={miou5:.4f} | FWIoU={fwiou5:.4f}")
for nm, v in zip(NAMES5, iou5):
    print(f"  {nm:<14} IoU={v:.4f}")


In [ ]:
# === Chart 2: IoU per-kelas (5-kelas, model final) ===
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(NAMES5, iou5, color=COLORS5, edgecolor="black", linewidth=0.5)
for b, v in zip(bars, iou5):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f"{v:.3f}", ha="center", fontsize=10)
ax.axhline(miou5, color="black", ls="--", lw=1.3, label=f"mIoU = {miou5:.3f}")
ax.set_ylabel("IoU"); ax.set_ylim(0, 1.05); ax.grid(axis="x", visible=False)
ax.set_title("IoU per Kelas - Model Final (5 kelas, TEST set)")
ax.legend(); plt.setp(ax.get_xticklabels(), rotation=15, ha="right")
fig.savefig(CHART_DIR / "02_iou_per_kelas.png"); plt.show()
print("Tersimpan:", CHART_DIR / "02_iou_per_kelas.png")


## Chart 3 — Confusion matrix 5 kelas (ternormalisasi)
Untuk analisis kesalahan di **Pembahasan**. Diagonal = benar; sel lain = tertukar.

In [ ]:
# === Chart 3: Confusion matrix 5-kelas (ternormalisasi per baris) ===
cmn = cm5 / cm5.sum(1, keepdims=True).clip(1)
fig, ax = plt.subplots(figsize=(6.6, 5.6))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=10,
                color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xticks(range(5)); ax.set_xticklabels(NAMES5, rotation=40, ha="right")
ax.set_yticks(range(5)); ax.set_yticklabels(NAMES5)
ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual (ground truth)")
ax.set_title("Confusion Matrix - 5 kelas (ternormalisasi per baris)")
ax.grid(False); fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.savefig(CHART_DIR / "03_confusion_matrix.png"); plt.show()
print("Tersimpan:", CHART_DIR / "03_confusion_matrix.png")


## Chart 4 — Panel metrik utama (5 kelas)
Angka headline untuk **Abstrak/Pembahasan**. mIoU dibedakan warna karena paling konservatif.

In [ ]:
# === Chart 4: Panel metrik utama (5-kelas) ===
labels = ["Overall\nAccuracy", "Kappa", "FWIoU", "mIoU"]
vals = [oa5, kappa5, fwiou5, miou5]
cols = ["#2E7D32", "#2E7D32", "#2E7D32", "#F9A825"]
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(labels, vals, color=cols, edgecolor="black", linewidth=0.5)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f"{v:.3f}",
            ha="center", fontsize=11, fontweight="bold")
ax.set_ylim(0, 1.08); ax.set_ylabel("Nilai"); ax.grid(axis="x", visible=False)
ax.set_title("Metrik Evaluasi Model - Skema Final 5 Kelas (TEST set)")
fig.savefig(CHART_DIR / "04_panel_metrik.png"); plt.show()
print("Tersimpan:", CHART_DIR / "04_panel_metrik.png")


## Chart 5 (opsional) — IoU 7 kelas: sebelum vs sesudah fine-tune
Cerita **pengembangan model** (fine-tune menaikkan tiap kelas, terutama kelas minoritas). Butuh `summary_finetune.json` (Bagian 2) -- dilewati otomatis kalau tak ada.

In [ ]:
# === Chart 5: IoU per-kelas 7-kelas, sebelum vs sesudah fine-tune ===
if iou7_before is None:
    print("Dilewati -- summary_finetune.json tak ditemukan (lihat Bagian 2).")
else:
    x = np.arange(7); w = 0.38
    fig, ax = plt.subplots(figsize=(10, 4.8))
    b1 = ax.bar(x - w / 2, iou7_before, w, label="Sebelum fine-tune",
                color="#B0BEC5", edgecolor="black", linewidth=0.4)
    b2 = ax.bar(x + w / 2, iou7_after, w, label="Sesudah fine-tune",
                color="#1565C0", edgecolor="black", linewidth=0.4)
    for bars in (b1, b2):
        for b in bars:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.012,
                    f"{b.get_height():.2f}", ha="center", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(NAMES7, rotation=20, ha="right")
    ax.set_ylabel("IoU"); ax.set_ylim(0, 1.08); ax.grid(axis="x", visible=False)
    ax.set_title("Pengembangan Model 7 Kelas: IoU Sebelum vs Sesudah Fine-tune (TEST)")
    ax.legend()
    fig.savefig(CHART_DIR / "05_7kelas_sebelum_sesudah.png"); plt.show()
    print("Tersimpan:", CHART_DIR / "05_7kelas_sebelum_sesudah.png")
